In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/post_cell_9.pickle

trying: ['nid']
me:  7
trying: ['data']
me:  1
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['idxs']
me:  5
trying: ['Path']
me:  0
trying: ['x']
me:  5
trying: ['pd']
me:  0
trying: ['plt']
me:  0
trying: ['scout']
me:  1
trying: ['players']
me:  1
trying: ['factor']
me:  1
trying: ['los']
me:  9
trying: ['benchmark_name']
me:  1
trying: ['gid']
me:  7
trying: ['plays']
me:  1
trying: ['orig_output']
me:  20
trying: ['df']
me:  17
trying: ['pid']
me:  7
trying: ['snap_speed']
me:  17
trying: ['sns']
me:  0
trying: ['time']
me:  0
trying: ['cp']
me:  0
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable Path
Declaring variable pd
Declaring variable plt
Declaring variable sns
Declaring variable time
Declaring variable cp
Declaring variable data
Declaring variable scout
Declaring variable players
Declaring variable factor
Declaring variable benchmark_name
Declaring variable plays
Declaring variable idxs
Declaring variable x
Declaring variable nid
Declaring variable gid
Declaring 

In [4]:
%%RecordEvent
%%time
### cell 10 ###

# cell 10 – further optimized for cuDF
# 1) one groupby to get both max (offense) and min (defense) los_diff per play/player
los_team = (
    df[['gameId','playId','nflId','posTeam','los_diff']]
      .groupby(['gameId','playId','nflId','posTeam'])
      .los_diff.agg(['max','min'])
      .reset_index()
)
# 2) pick the right value per posTeam: max for offense (1), min for defense (0)
los = (
    los_team
      .assign(los_diff=los_team['max'].where(los_team['posTeam']==1,
                                           los_team['min']))
      [['nflId','los_diff']]
)
# 3) average LOS diff per player
top = (
    los.groupby('nflId', as_index=False)
       .los_diff.mean()
)
# 4) merge back onto snap_speed and rename
snap_speed = (
    snap_speed
      .merge(top, on='nflId')
      .rename(columns={
          's':'snap_s',
          'a':'snap_a',
          'los_diff':'snap_los_diff'
      })
)

CPU times: user 52.6 ms, sys: 59 μs, total: 52.6 ms
Wall time: 52.6 ms


In [5]:
%Checkpoint /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_1.pickle

migration speed (bps): 798219007.0616919
---------------------------
variables to migrate:
los_team 32
idxs 144088
x 132776
df 32
Path 904
factor 24
los 32
pd 72
snap_speed 32
BENCHMARKS_TO_PATHS 2272
scout 32
players 32
benchmark_name 90
data 32
top 32
nid 28
plays 32
orig_output 32
plt 72
sns 72
pid 28
time 72
gid 32
cp 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['BENCHMARKS_TO_PATHS', 'Path']
Active variables ['plays', 'data']
Intermediate variables ['factor', 'players', 'benchmark_name', 'scout']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['data']
Active variables []
Intermediate variables []
Future variables ['plays', 'data']
Modified dataframes
======= Cell 2 =======
Input variables ['data']
Active variables ['df']
Intermediate variables ['idxs', 'x']
Future variables ['plays', 'data']
Modified dataframes
======= Cell 3 =======
Input variables ['df']
Active variables ['pid', 'nid', 'gid']
Intermediate variables []
Future variables ['plays', 'df', 'data']
Modified dataframes
======= Cell 4 =======
Input variables ['df', 'data']
Active variables ['df']
Intermediate variables ['los']
Future variables ['plays', 'pid', 'nid', 'gid']
Modified dataframes
======= Cell 5 =======
Input variables ['pid', 'nid', 'df', 'gid']
Active variables []
Intermediate variables []
Future 

In [7]:

with open("/home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/opt_cell_exec_info_10_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[10], f)


In [8]:
opt_output = Out.get(4)

In [9]:
snap_speed_opt = snap_speed
%LoadCheckpoint /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/post_cell_10.pickle
assert compare_df(snap_speed_opt, snap_speed)

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['sns']
me:  0
trying: ['pid']
me:  7
trying: ['time']
me:  0
trying: ['gid']
me:  7
trying: ['cp']
me:  0
trying: ['los_diff']
me:  21
trying: ['idxs']
me:  5
trying: ['x']
me:  5
trying: ['df']
me:  17
trying: ['Path']
me:  0
trying: ['factor']
me:  1
trying: ['off']
me:  21
trying: ['pd']
me:  0
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['scout']
me:  1
trying: ['benchmark_name']
me:  1
trying: ['players']
me:  1
trying: ['snap_speed']
me:  21
trying: ['data']
me:  1
trying: ['orig_output']
me:  22
trying: ['los']
me:  9
trying: ['def1']
me:  21
trying: ['nid']
me:  7
trying: ['plays']
me:  1
trying: ['plt']
me:  0
Declaring variable sns
Declaring variable time
Declaring variable cp
Declaring variable Path
Declaring variable pd
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable plt
Declaring variable factor
Declaring variable scout
Declaring variable benchmark_name
Declaring variable players
Declaring variable data
Declaring variable plays
Declaring variable idxs

ValueError: Shape mismatch: (223, 4) vs (222, 4)